# Exercise 2 – EfficientNet-B0 from Scratch

In [1]:
import json, torch, timm, numpy as np
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [2]:
with open('splits.json') as f:
    splits = json.load(f)
ds_raw = load_dataset("blanchon/UC_Merced")['train']

class UCMercedDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        item = ds_raw[self.indices[idx]]
        img = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = UCMercedDataset(splits['train'], train_tf)
val_ds   = UCMercedDataset(splits['val'],   val_tf)
test_ds  = UCMercedDataset(splits['test'],  val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 1470, Val: 315, Test: 315


In [3]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=21)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 4,034,449


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss/total, correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval(); total_loss = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss/total, correct/total

import os, pickle
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc = 0
EPOCHS = 20

if os.path.exists('scratch_best.pth') and os.path.exists('scratch_history.pkl'):
    print("Checkpoint found — loading pre-trained weights and history.")
    model.load_state_dict(torch.load('scratch_best.pth', map_location=device))
    with open('scratch_history.pkl','rb') as f: history = pickle.load(f)
    print(f"Loaded history for {len(history['train_loss'])} epochs.")
else:
    for epoch in range(1, EPOCHS+1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, device)
        history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc);   history['val_acc'].append(vl_acc)
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), 'scratch_best.pth')
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.suptitle('EfficientNet-B0 from Scratch'); plt.tight_layout()
plt.savefig('scratch_curves.png', dpi=100); plt.show()
import pickle
with open('scratch_history.pkl','wb') as f: pickle.dump(history, f)

In [ ]:
model.load_state_dict(torch.load('scratch_best.pth', map_location=device))
test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
print(f"Final Test Accuracy (Scratch): {test_acc:.4f} ({test_acc*100:.2f}%)")